<a href="https://colab.research.google.com/github/blskyue/AI-Research-Agent/blob/main/Gmail_RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Indexing

🛠️ Project Setup & Authentication
To run this agent, we need to handle two types of authentication:

IMAP Access: Used for fetching and indexing emails. You need to provide your Gmail Username and an App Password (stored securely in Colab Secrets or a .env file).

Gmail API: Used for the send_email tool. This requires a credentials.json file from the Google Cloud Console with the Gmail API enabled to authorize the agent to send messages on your behalf.

Environment Variables: The system retrieves these credentials using google.colab.userdata for enhanced security.

In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip install langchain-huggingface

In [ ]:
!pip install langchain-text-splitters

import imaplib, os
from dotenv import load_dotenv
from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv()

🧠 Data Indexing Pipeline (RAG)
This stage transforms raw emails into a searchable knowledge base:

Fetching: Emails are retrieved via IMAP, and the HTML content is cleaned using BeautifulSoup to extract clean text.

Chunking: The text is split into smaller, manageable pieces using the RecursiveCharacterTextSplitter to maintain context.

Embedding: Each chunk is converted into a numerical vector using HuggingFaceEmbeddings (sentence-transformers).

Vector Store: These vectors are stored in an InMemoryVectorStore, enabling the agent to perform fast "Similarity Search" based on meaning.

In [ ]:
def fetch_emails(n=1000):
    mail = imaplib.IMAP4_SSL('imap.gmail.com')
    mail.login(os.getenv('GMAIL_USERNAME'), os.getenv('GMAIL_PASSWORD'))
    mail.select('inbox')

    _, messages = mail.search(None, "ALL")
    email_ids = messages[0].split()[-n:]

    parsed = []
    for eid in email_ids:
        _, msg_data = mail.fetch(eid, "(RFC822)")
        raw_email = msg_data[0][1]
        parsed.append(BytesParser(policy=policy.default).parsebytes(raw_email))

    mail.logout()
    return parsed

In [ ]:
def convert_email(msg):
    body = ""

    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body += part.get_content()
            elif part.get_content_type() == "text/html":
                soup = BeautifulSoup(part.get_content(), "html.parser")
                body += soup.get_text()
    else:
        body = msg.get_content()

    return Document(
        page_content=body.strip(),
        metadata={
            "subject": msg.get("subject", ""),
            "sender": msg.get("from", ""),
            "date": msg.get("date", ""),
        }
    )

In [ ]:
from google.colab import userdata
import os

# جلب البيانات من Secrets
email_user = userdata.get('GMAIL_USERNAME')
email_pass = userdata.get('GMAIL_PASSWORD')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# تأكد أن القيم ليست فارغة قبل المتابعة
if email_user is None or email_pass is None:
    print("خطأ: تأكد من تفعيل Notebook access للمفاتيح في قائمة Secrets")
else:
    print("تم تحميل جميع المفاتيح بنجاح!")

In [ ]:
import os
os.environ['GMAIL_USERNAME'] = email_user
os.environ['GMAIL_PASSWORD'] = email_pass

# Load emails
parsed_emails = fetch_emails()

# Convert to documents
docs = [convert_email(m) for m in parsed_emails if m]

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = splitter.split_documents(docs)

# Embed + store
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(splits)

#### RAG Agnet

In [ ]:
!pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install langchain-google-community

In [ ]:
!pip install langchain-openai
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_google_community.gmail.utils import (
    build_gmail_service,
    get_google_credentials,
)
from email.mime.text import MIMEText
import base64

checkpointer = InMemorySaver()

In [ ]:
SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.send"
]

credentials = get_google_credentials(
    token_file="token.json",
    scopes=SCOPES,
    client_secrets_file="credentials.json",
)

🤖 AI Agent Reasoning & Tools
The agent is built using the ReAct (Reasoning and Acting) framework, allowing it to think and choose the right tool based on the user's request:

Search Tool (search_emails): When the user asks a question about their history, the agent queries the vector store to find relevant information.

Action Tool (send_email): If the user requests to send a response or a new message, the agent invokes the Gmail API tool.

Memory (Checkpointer): Using InMemorySaver, the agent maintains a Thread ID, allowing it to remember previous parts of the conversation and handle follow-up questions accurately.

In [ ]:
llm = ChatOpenAI(
    model="anthropic/claude-opus-4.6",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=500
)

@tool
def search_emails(query: str) -> str:
    """Search relevant information from emails."""
    docs = vector_store.similarity_search(query, k=3)

    return "\n\n".join(
        f"Subject: {d.metadata['subject']}\n"
        f"From: {d.metadata['sender']}\n"
        f"Content: {d.page_content[:300]}"
        for d in docs
    )

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email via Gmail API."""

    message = MIMEText(body)
    message["to"] = to
    message["subject"] = subject

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()

    request = {
        "raw": raw
    }

    service = build_gmail_service(credentials=credentials)

    sent_message = service.users().messages().send(
        userId="me",
        body=request
    ).execute()

    return f"Email sent successfully. Message ID: {sent_message['id']}"

agent = create_agent(
    model=llm,
    tools=[search_emails, send_email],
    system_prompt=(
        "You are an assistant that answers questions using email data. "
        "Use the search_emails tool OR the send_email tool when needed."
    ),
    checkpointer=checkpointer,
)

In [ ]:
config: RunnableConfig = {
    "configurable": {
        "thread_id": "1"
    }
}

query = "Did I receive any invitations?"

result1 = agent.invoke(
    input={"messages": [HumanMessage(query)]},
    config=config,
)

for msg in result1["messages"]:
    msg.pretty_print()

In [ ]:
query2 = "No just tell me how many invitations I received?"

result2 = agent.invoke(
    input={"messages": [HumanMessage(query2)]},
    config=config,
)

for msg in result2["messages"]:
    msg.pretty_print()

In [ ]:
config2: RunnableConfig = {
    "configurable": {
        "thread_id": "3"
    }
}

query3 = "send an email to nourah0905@gmail.com and say Hi"

result3 = agent.invoke(
    input={"messages": [HumanMessage(query3)]},
    config=config2,
)

for msg in result3["messages"]:
    msg.pretty_print()